In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/application_train.csv')
print("Original shape:", df.shape)

Original shape: (307511, 122)


In [2]:
# Complete missing value report
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_%': missing_pct
}).sort_values('missing_%', ascending=False)

# Split into categories
high_missing   = missing_df[missing_df['missing_%'] > 40]
medium_missing = missing_df[(missing_df['missing_%'] > 0) & (missing_df['missing_%'] <= 40)]
no_missing     = missing_df[missing_df['missing_%'] == 0]

print(f"Columns with >40% missing:  {len(high_missing)}")
print(f"Columns with 1-40% missing: {len(medium_missing)}")
print(f"Columns with 0% missing:    {len(no_missing)}")
print()
print("=== HIGH MISSING (>40%) — candidates for dropping ===")
print(high_missing.to_string())

Columns with >40% missing:  49
Columns with 1-40% missing: 15
Columns with 0% missing:    58

=== HIGH MISSING (>40%) — candidates for dropping ===
                              missing_count  missing_%
COMMONAREA_AVG                       214865      69.87
COMMONAREA_MODE                      214865      69.87
COMMONAREA_MEDI                      214865      69.87
NONLIVINGAPARTMENTS_MEDI             213514      69.43
NONLIVINGAPARTMENTS_MODE             213514      69.43
NONLIVINGAPARTMENTS_AVG              213514      69.43
FONDKAPREMONT_MODE                   210295      68.39
LIVINGAPARTMENTS_AVG                 210199      68.35
LIVINGAPARTMENTS_MEDI                210199      68.35
LIVINGAPARTMENTS_MODE                210199      68.35
FLOORSMIN_MODE                       208642      67.85
FLOORSMIN_AVG                        208642      67.85
FLOORSMIN_MEDI                       208642      67.85
YEARS_BUILD_AVG                      204488      66.50
YEARS_BUILD_MODE           

In [3]:
# Smart dropping — only drop building/property columns, keep important ones
cols_to_keep_despite_missing = ['EXT_SOURCE_1', 'OWN_CAR_AGE']

# All columns with >40% missing
high_missing_cols = missing_df[missing_df['missing_%'] > 40].index.tolist()

# Remove the ones we decided to keep
cols_to_drop = [c for c in high_missing_cols if c not in cols_to_keep_despite_missing]

print(f"Total high-missing columns:  {len(high_missing_cols)}")
print(f"Decided to keep:             {len(cols_to_keep_despite_missing)}")
print(f"Actually dropping:           {len(cols_to_drop)}")
print()
print("Keeping despite high missing:")
for col in cols_to_keep_despite_missing:
    pct = missing_df.loc[col, 'missing_%']
    print(f"  {col}: {pct}% missing — kept because high predictive value")

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nShape after dropping: {df.shape}")

Total high-missing columns:  49
Decided to keep:             2
Actually dropping:           47

Keeping despite high missing:
  EXT_SOURCE_1: 56.38% missing — kept because high predictive value
  OWN_CAR_AGE: 65.99% missing — kept because high predictive value

Shape after dropping: (307511, 75)


In [4]:
print("DAYS_EMPLOYED value counts (top 10):")
print(df['DAYS_EMPLOYED'].describe())
print()
print("Suspiciously large positive value count:")
print((df['DAYS_EMPLOYED'] == 365243).sum())

DAYS_EMPLOYED value counts (top 10):
count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

Suspiciously large positive value count:
55374


In [5]:
# DAYS_BIRTH, DAYS_REGISTRATION etc. are stored as negative numbers
# (days before application date). Convert to positive for interpretability.
days_cols = ['DAYS_BIRTH', 'DAYS_REGISTRATION', 
             'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']

for col in days_cols:
    if col in df.columns:
        df[col] = df[col].abs()

print("Converted negative DAYS columns to positive:")
print(df[days_cols].describe())

Converted negative DAYS columns to positive:
          DAYS_BIRTH  DAYS_REGISTRATION  DAYS_ID_PUBLISH  \
count  307511.000000      307511.000000    307511.000000   
mean    16036.995067        4986.120328      2994.202373   
std      4363.988632        3522.886321      1509.450419   
min      7489.000000           0.000000         0.000000   
25%     12413.000000        2010.000000      1720.000000   
50%     15750.000000        4504.000000      3254.000000   
75%     19682.000000        7479.500000      4299.000000   
max     25229.000000       24672.000000      7197.000000   

       DAYS_LAST_PHONE_CHANGE  
count           307510.000000  
mean               962.858788  
std                826.808487  
min                  0.000000  
25%                274.000000  
50%                757.000000  
75%               1570.000000  
max               4292.000000  


In [6]:
print("CODE_GENDER unique values:")
print(df['CODE_GENDER'].value_counts())

CODE_GENDER unique values:
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64


In [7]:
# Replace XNA with the mode (most common gender)
df['CODE_GENDER'].replace('XNA', df['CODE_GENDER'].mode()[0], inplace=True)
print("After fix:")
print(df['CODE_GENDER'].value_counts())

After fix:
CODE_GENDER
F    202452
M    105059
Name: count, dtype: int64


C:\Users\shaz9\AppData\Local\Temp\ipykernel_22852\2428883144.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['CODE_GENDER'].replace('XNA', df['CODE_GENDER'].mode()[0], inplace=True)


In [8]:
# Separate columns by type
numeric_cols     = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from numeric list
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

print(f"Numeric columns to impute:     {len(numeric_cols)}")
print(f"Categorical columns to impute: {len(categorical_cols)}")

# Check which still have missing values
print("\nNumeric cols still missing:")
num_missing = df[numeric_cols].isnull().sum()
print(num_missing[num_missing > 0])

print("\nCategorical cols still missing:")
cat_missing = df[categorical_cols].isnull().sum()
print(cat_missing[cat_missing > 0])

Numeric columns to impute:     62
Categorical columns to impute: 12

Numeric cols still missing:
AMT_ANNUITY                       12
AMT_GOODS_PRICE                  278
OWN_CAR_AGE                   202929
CNT_FAM_MEMBERS                    2
EXT_SOURCE_1                  173378
EXT_SOURCE_2                     660
EXT_SOURCE_3                   60965
OBS_30_CNT_SOCIAL_CIRCLE        1021
DEF_30_CNT_SOCIAL_CIRCLE        1021
OBS_60_CNT_SOCIAL_CIRCLE        1021
DEF_60_CNT_SOCIAL_CIRCLE        1021
DAYS_LAST_PHONE_CHANGE             1
AMT_REQ_CREDIT_BUREAU_HOUR     41519
AMT_REQ_CREDIT_BUREAU_DAY      41519
AMT_REQ_CREDIT_BUREAU_WEEK     41519
AMT_REQ_CREDIT_BUREAU_MON      41519
AMT_REQ_CREDIT_BUREAU_QRT      41519
AMT_REQ_CREDIT_BUREAU_YEAR     41519
dtype: int64

Categorical cols still missing:
NAME_TYPE_SUITE     1292
OCCUPATION_TYPE    96391
dtype: int64


In [9]:
# --- Special case imputations first ---

# OWN_CAR_AGE: missing = no car, so fill with 0
df['OWN_CAR_AGE'].fillna(0, inplace=True)

# AMT_REQ_CREDIT_BUREAU columns: missing = no queries made, fill with 0
bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR'
]
for col in bureau_cols:
    df[col].fillna(0, inplace=True)

# OCCUPATION_TYPE: missing = information not provided, preserve as 'Unknown'
df['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)

print("Special case imputations done.")

# --- Standard median imputation for remaining numeric columns ---
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  {col}: filled with median = {median_val:.4f}")

# --- Mode imputation for remaining categorical columns ---
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  {col}: filled with mode = '{mode_val}'")

print("\nImputation complete.")
print("Remaining missing values:", df.isnull().sum().sum())

C:\Users\shaz9\AppData\Local\Temp\ipykernel_22852\4257755911.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['OWN_CAR_AGE'].fillna(0, inplace=True)
C:\Users\shaz9\AppData\Local\Temp\ipykernel_22852\4257755911.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

Special case imputations done.


C:\Users\shaz9\AppData\Local\Temp\ipykernel_22852\4257755911.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
C:\Users\shaz9\AppData\Local\Temp\ipykernel_22852\4257755911.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

  AMT_ANNUITY: filled with median = 24903.0000
  AMT_GOODS_PRICE: filled with median = 450000.0000
  CNT_FAM_MEMBERS: filled with median = 2.0000
  EXT_SOURCE_1: filled with median = 0.5060
  EXT_SOURCE_2: filled with median = 0.5660
  EXT_SOURCE_3: filled with median = 0.5353
  OBS_30_CNT_SOCIAL_CIRCLE: filled with median = 0.0000
  DEF_30_CNT_SOCIAL_CIRCLE: filled with median = 0.0000
  OBS_60_CNT_SOCIAL_CIRCLE: filled with median = 0.0000
  DEF_60_CNT_SOCIAL_CIRCLE: filled with median = 0.0000
  DAYS_LAST_PHONE_CHANGE: filled with median = 757.0000


C:\Users\shaz9\AppData\Local\Temp\ipykernel_22852\4257755911.py:36: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(mode_val, inplace=True)


  NAME_TYPE_SUITE: filled with mode = 'Unaccompanied'

Imputation complete.
Remaining missing values: 0


In [10]:
# Check key financial columns for outliers
cols_to_check = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 
                 'AMT_ANNUITY', 'DAYS_EMPLOYED']

for col in cols_to_check:
    if col in df.columns:
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
        print(f"{col}: {outliers} outliers ({outliers/len(df)*100:.2f}%)")

AMT_INCOME_TOTAL: 14035 outliers (4.56%)
AMT_CREDIT: 6562 outliers (2.13%)
AMT_ANNUITY: 7504 outliers (2.44%)
DAYS_EMPLOYED: 72217 outliers (23.48%)


In [11]:
# Cap outliers using 1st and 99th percentile (Winsorization)
# We cap instead of drop — dropping rows loses applicant data entirely
# Capping preserves the row but limits extreme values' influence

cols_to_cap = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']

for col in cols_to_cap:
    if col in df.columns:
        lower_cap = df[col].quantile(0.01)
        upper_cap = df[col].quantile(0.99)
        df[col] = df[col].clip(lower=lower_cap, upper=upper_cap)
        print(f"{col} capped at [{lower_cap:.0f}, {upper_cap:.0f}]")

AMT_INCOME_TOTAL capped at [45000, 472500]
AMT_CREDIT capped at [76410, 1854000]
AMT_ANNUITY capped at [6183, 70006]


In [12]:
print("Duplicate rows before:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Duplicate rows after:", df.duplicated().sum())
print("Shape after deduplication:", df.shape)

Duplicate rows before: 0
Duplicate rows after: 0
Shape after deduplication: (307511, 75)


In [13]:
print("=== FINAL CLEANED DATASET ===")
print(f"Shape: {df.shape}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"Target distribution:\n{df['TARGET'].value_counts()}")
print(f"\nMemory usage: {df.memory_usage().sum() / 1024**2:.1f} MB")

# Save cleaned data
df.to_csv('../data/processed/loan_cleaned.csv', index=False)
print("\nSaved to data/processed/loan_cleaned.csv")

=== FINAL CLEANED DATASET ===
Shape: (307511, 75)
Missing values remaining: 0
Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Memory usage: 176.0 MB

Saved to data/processed/loan_cleaned.csv
